In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [47]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."
system_message += "You can help the customer when he wants to know the ticket prices."
system_message += "Moreover, you can also help them to see if there are any tickets with discount offers."
system_message += "Lastly, you can help them book the flight and display the final booking details to the user."

In [49]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

discount_function = {
    "name": "get_ticket_discount",
    "description": "Get the discount of a return ticket to the destination city. Call this whenever you need to know the available discounts for a particular destination, for example when a customer asks 'How much is the discount on this ticket to this city', 'What are the available discounts you offer'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

booking_function = {
    "name": "make_booking",
    "description": "Make the flight booking for a particular destination city. Call this whenever you need to make the flight booking for a particular destination, for example when a customer says 'Book me a flight to Paris', 'I would like to book my tickets to Berlin'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "num_tickets": {
                "type": "integer",
                "description": "The number of tickets to book",
            },
            "mail_address": {
                "type": "string",
                "description": "Mail address to send the ticket to",
            },
        },
        "required": ["destination_city", "num_tickets", "mail_address"],
        "additionalProperties": False
    }
}

In [69]:
import base64
from io import BytesIO
from PIL import Image

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
ticket_discounts = {"london": 15, "paris": 10}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")
    
def get_ticket_discount(destination_city):
    print(f"Tool get_ticket_discount called for {destination_city}")
    city = destination_city.lower()
    return ticket_discounts.get(city, 0)
    
def make_booking(destination_city, num_tickets):
    print(f"Tool make_booking called for {destination_city}")
    city = destination_city.lower()
    if ticket_prices.get(destination_city) == "Unknown":
        return None
    price = int(get_ticket_price(destination_city).replace("$", ""))
    total_price = price * num_tickets
    discount = ticket_discounts.get(city, 0)
    if discount > 0:
        total_price = total_price - (total_price*discount)/100
    return total_price
    
def generate_image(destination_city):
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=f"An image representing a vacation in {destination_city}, showing tourist spots and everything unique about {destination_city}, in a vibrant pop-art style",
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))
    
            # # Generate a unique booking ID
            # booked_ID = generate_unique_booking_id()
            # # Define the booking data
            # data = {
            #     "booking_id": [booked_ID],
            #     "mail_address": [mail_address],
            #     "destination_city": [destination_city],
            #     "num_tickets": [num_tickets],
            #     "ticket_class": [ticket_class],
            #     "total_price": [total_price],
            # }
            # booking_temp = pd.DataFrame(data)
            # bookingDB = bookingDB._append(booking_temp)
            # # Can update with real booking system
            # bookingDB.to_csv('bookingDB.csv', index=False)    

In [48]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function},
         {"type": "function", "function": discount_function},
         {"type": "function", "function": booking_function}]

In [75]:
def chat(history):
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    image = None
    
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        messages.append(message)
        tool_responses = handle_tool_call(message)
        for tool_response in tool_responses:
            print(tool_response)
            messages.append(tool_response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        for tool_call in message.tool_calls:
            if tool_call.function.name == "make_booking":
                arguments = json.loads(tool_call.function.arguments)
                destination_city = arguments.get('destination_city')
                image = generate_image(destination_city)
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]
    return history, image

In [76]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    global bookingDB
    # print(message.tool_calls)
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        destination_city = arguments.get('destination_city')
        if tool_call.function.name == "get_ticket_price":
            price = get_ticket_price(destination_city)
            response = {
                "role": "tool",
                "content": json.dumps({"destination_city": destination_city,"price": price}),
                "tool_call_id": tool_call.id
            }
        elif tool_call.function.name == "get_ticket_discount":
            discount = get_ticket_discount(destination_city)
            response = {
                "role": "tool",
                "content": json.dumps({"destination_city": destination_city,"discount": discount}),
                "tool_call_id": tool_call.id
            }
        elif tool_call.function.name == "make_booking":
            num_tickets = arguments.get('num_tickets')
            mail_address = arguments.get('mail_address')
            total_price = make_booking(destination_city, num_tickets)
            response = {
                "role": "tool",
                "content": json.dumps({"destination_city": destination_city,"num_tickets": num_tickets, "mail_address": mail_address, "total_price": total_price}),
                "tool_call_id": tool_call.id
            }
        responses.append(response)
    return responses

In [55]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7878

To create a public link, set `share=True` in `launch()`.


[ChatCompletionMessageToolCall(id='call_mOxS12RjfAgkuG17q1yDl9e8', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_discount'), type='function')]
Tool get_ticket_discount called for Tokyo
[ChatCompletionMessageToolCall(id='call_3i7tcy6Ysc7rdhRYX1nqCv1k', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function')]
Tool get_ticket_price called for Tokyo
[ChatCompletionMessageToolCall(id='call_dZ6qQ4KyubtJ2zqsolxvXoao', function=Function(arguments='{"destination_city":"Tokyo","num_tickets":3,"mail_address":"jain@gmail.com"}', name='make_booking'), type='function')]
Tool make_booking called for Tokyo
Tool get_ticket_price called for Tokyo
[ChatCompletionMessageToolCall(id='call_6coYdJ0yVfvUjTb3GeFmMfik', function=Function(arguments='{"destination_city": "Paris"}', name='get_ticket_price'), type='function'), ChatCompletionMessageToolCall(id='call_V89Bf1PvB0SHmp0dawEgtLn0', function=Function(arguments='{"destination_city

In [77]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant:")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        history += [{"role":"user", "content":message}]
        return "", history

    entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )
    clear.click(lambda: None, inputs=None, outputs=chatbot, queue=False)
ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7885

To create a public link, set `share=True` in `launch()`.


Tool get_ticket_price called for Paris
Tool get_ticket_discount called for Paris
{'role': 'tool', 'content': '{"destination_city": "Paris", "price": "$899"}', 'tool_call_id': 'call_oBEaobpQTSr6LWaRHdWYsQ6f'}
{'role': 'tool', 'content': '{"destination_city": "Paris", "discount": 10}', 'tool_call_id': 'call_QogANPWZiSwT4P3Tny6PaKjB'}
Tool make_booking called for Paris
Tool get_ticket_price called for Paris
{'role': 'tool', 'content': '{"destination_city": "Paris", "num_tickets": 2, "mail_address": "jain@gmail.com", "total_price": 1618.2}', 'tool_call_id': 'call_scSO8kQ1CoqbQaGTFmDMHjbY'}
Tool make_booking called for Paris
Tool get_ticket_price called for Paris
{'role': 'tool', 'content': '{"destination_city": "Paris", "num_tickets": 2, "mail_address": "jain@gmail.com", "total_price": 1618.2}', 'tool_call_id': 'call_a54jnWEuB7xldiXUwyDoIYOA'}
